# Regional Comparison: Terrain Regions in P-Adic Space

Analyze how different terrain regions (low/high elevation) map to the p-adic Sierpinski structure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

cache_dir = Path('/Volumes/Fangorn/padic_fractal_analysis/cache')

dem_data = np.load(str(cache_dir / 'dem_clean.npy'))
print(f"DEM shape: {dem_data.shape}")

## Elevation Extremes in Embedding Space

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, scale_name in enumerate(['l=4', 'l=6']):
    embeddings = np.load(str(cache_dir / f'padic_embeddings_{scale_name}.npy'))
    grid_values = np.load(str(cache_dir / f'dem_grid_{scale_name}.npy'))
    elevations = grid_values.flatten()
    
    n_extremes = max(3, len(elevations) // 10)
    
    # Identify extremes
    low_idx = np.argsort(elevations)[:n_extremes]
    high_idx = np.argsort(elevations)[-n_extremes:]
    mid_idx = np.argsort(np.abs(elevations - np.median(elevations)))[:n_extremes]
    
    # Plot 1: Elevation in color
    ax = axes[0, idx]
    scatter = ax.scatter(embeddings[:, 0], embeddings[:, 1],
                         c=elevations, cmap='viridis', s=100, alpha=0.6)
    ax.scatter(embeddings[low_idx, 0], embeddings[low_idx, 1],
              s=200, marker='^', edgecolors='red', linewidth=2, facecolors='none', label='Low elevation')
    ax.scatter(embeddings[high_idx, 0], embeddings[high_idx, 1],
              s=200, marker='v', edgecolors='blue', linewidth=2, facecolors='none', label='High elevation')
    ax.set_title(f'{scale_name}: Elevation Extremes', fontweight='bold')
    ax.set_xlabel('Real')
    ax.set_ylabel('Imag')
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='Elevation')
    
    # Plot 2: Clustering analysis
    ax = axes[1, idx]
    
    # Distances from origin
    distances = np.sqrt(embeddings[:, 0]**2 + embeddings[:, 1]**2)
    
    ax.scatter(elevations, distances, alpha=0.6, s=80, label='All regions')
    ax.scatter(elevations[low_idx], distances[low_idx], s=150, 
              marker='^', color='red', edgecolors='darkred', linewidth=2, label='Low elevation', zorder=5)
    ax.scatter(elevations[high_idx], distances[high_idx], s=150,
              marker='v', color='blue', edgecolors='darkblue', linewidth=2, label='High elevation', zorder=5)
    
    ax.set_xlabel('Elevation (m)')
    ax.set_ylabel('Distance from Origin in Embedding')
    ax.set_title(f'{scale_name}: Elevation vs Embedding Distance', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

## Regional Clustering Statistics

In [ ]:
print("\nRegional Clustering Analysis:")
print("="*50)

for scale_name in ['l=4', 'l=6']:
    embeddings = np.load(str(cache_dir / f'padic_embeddings_{scale_name}.npy'))
    grid_values = np.load(str(cache_dir / f'dem_grid_{scale_name}.npy'))
    elevations = grid_values.flatten()
    
    n_extremes = max(3, len(elevations) // 10)
    
    low_idx = np.argsort(elevations)[:n_extremes]
    high_idx = np.argsort(elevations)[-n_extremes:]
    
    # Compute intra-group distances
    def mean_distance(points):
        if len(points) <= 1:
            return 0
        dists = []
        for i in range(len(points)):
            for j in range(i+1, len(points)):
                d = np.sqrt(np.sum((points[i] - points[j])**2))
                dists.append(d)
        return np.mean(dists)
    
    low_dist = mean_distance(embeddings[low_idx])
    high_dist = mean_distance(embeddings[high_idx])
    
    print(f"\n{scale_name}:")
    print(f"  Low elevation regions:")
    print(f"    Count: {len(low_idx)}")
    print(f"    Mean intra-group distance: {low_dist:.4f}")
    print(f"    Spread: {'scattered' if low_dist > 0.5 else 'clustered'}")
    print(f"  High elevation regions:")
    print(f"    Count: {len(high_idx)}")
    print(f"    Mean intra-group distance: {high_dist:.4f}")
    print(f"    Spread: {'scattered' if high_dist > 0.5 else 'clustered'}")
    
    if low_dist > 0.5 and high_dist > 0.5:
        print(f"  → Elevation extremes are NOT clustered in embedding space")
        print(f"    This suggests p-adic structure is independent of elevation ordering")
    else:
        print(f"  → Elevation extremes show some clustering in embedding space")